In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

import base64
from io import BytesIO
from PIL import Image

from pydantic import BaseModel, Field
from system_prompts import PACKAGE_MANAGER_SYSTEM_PROMPT_V5 as PACKAGE_MANAGER_SYSTEM_PROMPT

from schemas import FlatResponse, Response, unflatten

In [2]:
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# OLLAMA_BASE_URL = "http://localhost:11434/v1"

# client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
item1 = {
    "id": 1,
    "color": "green",
    "name": "Dispenser",
    "weight": 4,
    "dimensions": {
        "x": 9.5,
        "y": 3.25,
        "z": 4.5
    },
    "quantity": 1
}

item2 = {
    "id": 2,
    "color": "blue",
    "name": "Tape",
    "weight": 0.3,
    "dimensions": {
        "x": 4.75,
        "y": 4.75,
        "z": 1
    },
    "quantity": 20
}

item3 = {
    "id": 3,
    "color": "red",
    "name": "Stand",
    "weight": 17,
    "dimensions": {
        "x": 28,
        "y": 4.38,
        "z": 23.38
    },
    "quantity": 3
}

items = [item1, item2, item3]

box1 = {
    "weightMax": 65,
    "id": 1,
    "name": "Small",
    "dimensions": {
        "x": 12,
        "y": 6,
        "z": 6.75
    }
}

box2 = {
    "weightMax": 80,
    "id": 2,
    "name": "Medium",
    "dimensions": {
        "x": 14.5,
        "y": 6.75,
        "z": 12.75
    }
}
box3 = {
    "weightMax": 70,
    "id": 3,
    "name": "Wide Low",
    "dimensions": {
        "x": 12,
        "y": 12,
        "z": 4.25
    } 
}
boxes = [box1, box2, box3]

In [ ]:
# Golf order 9112069 (from a Paccurate request). refIds kept as ids.
# Running this cell replaces the sample items/boxes above.
# Not carried over: compact-pattern rules (nesting isn't modelled) and the
# FedEx rate tables. lock-orientation on Navy Hat = fixed_orientation below.
from settings import PackingSettings

items = [
    {"id": 7703022, "name": "Gray Gloves", "color": "gray", "weight": 0.13,
     "dimensions": {"x": 0.63, "y": 4.13, "z": 10.63}, "quantity": 2},
    {"id": 7704701, "name": "White Towel", "color": "lightgray", "weight": 0.65,
     "dimensions": {"x": 3, "y": 8.2, "z": 6.5}, "quantity": 2},
    {"id": 7763901, "name": "Black Radar", "color": "black", "weight": 0.24,
     "dimensions": {"x": 4.3, "y": 7.8, "z": 7.5}, "quantity": 1},
    {"id": 7760001, "name": "Navy Hat", "color": "navy", "weight": 0.24,
     "dimensions": {"x": 4.3, "y": 7.8, "z": 7.5}, "quantity": 2},
    {"id": 7757501, "name": "White Visor", "color": "khaki", "weight": 0.24,
     "dimensions": {"x": 3.5, "y": 8, "z": 7}, "quantity": 2},
]

boxes = [
    {"id": 1, "name": "CASE", "weightMax": 65, "dimensions": {"x": 11, "y": 12, "z": 18.8}},
    {"id": 2, "name": "4PBO", "weightMax": 65, "dimensions": {"x": 13.75, "y": 13.5, "z": 17.25}},
    {"id": 3, "name": "BAGC", "weightMax": 65, "dimensions": {"x": 7, "y": 13, "z": 37}},
    {"id": 4, "name": "X3", "weightMax": 65, "dimensions": {"x": 6.75, "y": 16, "z": 21.25}},
    {"id": 5, "name": "X4", "weightMax": 65, "dimensions": {"x": 13.25, "y": 16, "z": 21.75}},
]

golf_settings = PackingSettings(fixed_orientation=[7760001])  # Navy Hat: lock-orientation

In [ ]:
# Golf ball order 9172381 (from a Paccurate request). refIds kept as ids.
# Running this cell replaces the items/boxes above.
# Cleaned up: dropped the all-zero placeholder box ("name") and the exact
# duplicates of DDZ and DZ35 (6DZ has DZ35's size but its own name: kept).
# The request's rules are about items not in this order, so none apply.
items = [
    {"id": 7187701, "name": "Practice", "color": "orange", "weight": 1.57,
     "dimensions": {"x": 1.9, "y": 5.4, "z": 7.6}, "quantity": 7},
    {"id": 7175201, "name": "Tour Response", "color": "steelblue", "weight": 1.53,
     "dimensions": {"x": 1.9, "y": 5.4, "z": 7.6}, "quantity": 9},
    {"id": 7154901, "name": "Junior", "color": "seagreen", "weight": 1.54,
     "dimensions": {"x": 1.9, "y": 5.4, "z": 7.6}, "quantity": 8},
]

boxes = [
    {"id": 1, "name": "CASE", "weightMax": 65, "dimensions": {"x": 11, "y": 12, "z": 18.8}},
    {"id": 2, "name": "SDZ", "weightMax": 65, "dimensions": {"x": 2.25, "y": 5.75, "z": 7.75}},
    {"id": 3, "name": "DDZ", "weightMax": 65, "dimensions": {"x": 4.25, "y": 5.75, "z": 8}},
    {"id": 4, "name": "DZ35", "weightMax": 65, "dimensions": {"x": 5.75, "y": 7.75, "z": 11.5}},
    {"id": 5, "name": "6DZ", "weightMax": 65, "dimensions": {"x": 5.75, "y": 7.75, "z": 11.5}},
    {"id": 6, "name": "2PBO", "weightMax": 65, "dimensions": {"x": 8.6, "y": 13.1, "z": 14}},
    {"id": 7, "name": "3PBO", "weightMax": 65, "dimensions": {"x": 8.75, "y": 13.5, "z": 21}},
    {"id": 8, "name": "4PBO", "weightMax": 65, "dimensions": {"x": 13.75, "y": 13.5, "z": 17.25}},
    {"id": 9, "name": "BAGC", "weightMax": 65, "dimensions": {"x": 7, "y": 13, "z": 37}},
    {"id": 10, "name": "IBOX", "weightMax": 65, "dimensions": {"x": 4.3, "y": 4.75, "z": 46}},
    {"id": 11, "name": "ISET", "weightMax": 65, "dimensions": {"x": 4.3, "y": 4.75, "z": 46}},
    {"id": 12, "name": "2KIT", "weightMax": 65, "dimensions": {"x": 5.2, "y": 5.5, "z": 50}},
    {"id": 13, "name": "RBOX", "weightMax": 65, "dimensions": {"x": 5.4, "y": 5.5, "z": 46.25}},
    {"id": 14, "name": "MBOX", "weightMax": 65, "dimensions": {"x": 5.2, "y": 7.5, "z": 47.25}},
    {"id": 15, "name": "ABOX", "weightMax": 65, "dimensions": {"x": 4.2, "y": 9.5, "z": 47.2}},
    {"id": 16, "name": "MUIR", "weightMax": 65, "dimensions": {"x": 12, "y": 10.2, "z": 41.9}},
    {"id": 17, "name": "TBOX", "weightMax": 65, "dimensions": {"x": 7.75, "y": 14, "z": 47.5}},
    {"id": 18, "name": "JBOX", "weightMax": 65, "dimensions": {"x": 11.25, "y": 14.75, "z": 47.25}},
    {"id": 19, "name": "UMBX", "weightMax": 65, "dimensions": {"x": 8, "y": 4.4, "z": 42}},
    {"id": 20, "name": "WM4P", "weightMax": 65, "dimensions": {"x": 6.63, "y": 7.19, "z": 9.88}},
    {"id": 21, "name": "2PKG", "weightMax": 65, "dimensions": {"x": 4.75, "y": 7.75, "z": 11.5}},
    {"id": 22, "name": "X1", "weightMax": 65, "dimensions": {"x": 6.5, "y": 11.75, "z": 9.25}},
    {"id": 23, "name": "B4", "weightMax": 65, "dimensions": {"x": 6.75, "y": 11.5, "z": 15.25}},
    {"id": 24, "name": "X2", "weightMax": 65, "dimensions": {"x": 6.75, "y": 9.75, "z": 23}},
    {"id": 25, "name": "X3", "weightMax": 65, "dimensions": {"x": 6.75, "y": 16, "z": 21.25}},
    {"id": 26, "name": "X4", "weightMax": 65, "dimensions": {"x": 13.25, "y": 16, "z": 21.75}},
    {"id": 27, "name": "6DDZ", "weightMax": 65, "dimensions": {"x": 8.25, "y": 11.5, "z": 11.75}},
]

In [ ]:
# Sample order A: home goods (mixed sizes; should fit one box)
items = [
    {"id": 101, "name": "Mug", "color": "teal", "weight": 0.9,
     "dimensions": {"x": 4.5, "y": 3.75, "z": 3.75}, "quantity": 4},
    {"id": 102, "name": "Dinner Plate", "color": "ivory", "weight": 1.6,
     "dimensions": {"x": 1.25, "y": 11, "z": 11}, "quantity": 4},
    {"id": 103, "name": "Candle Jar", "color": "goldenrod", "weight": 1.1,
     "dimensions": {"x": 4, "y": 3.5, "z": 3.5}, "quantity": 3},
    {"id": 104, "name": "Cookbook", "color": "firebrick", "weight": 2.4,
     "dimensions": {"x": 1.5, "y": 8, "z": 10}, "quantity": 1},
    {"id": 105, "name": "Kitchen Towel Set", "color": "lightblue", "weight": 0.7,
     "dimensions": {"x": 2.5, "y": 6, "z": 8}, "quantity": 2},
]

boxes = [
    {"id": 1, "name": "S-10", "weightMax": 40, "dimensions": {"x": 6, "y": 10, "z": 10}},
    {"id": 2, "name": "M-12", "weightMax": 50, "dimensions": {"x": 8, "y": 12, "z": 12}},
    {"id": 3, "name": "L-14", "weightMax": 60, "dimensions": {"x": 10, "y": 14, "z": 14}},
    {"id": 4, "name": "Flat-16", "weightMax": 50, "dimensions": {"x": 6, "y": 12, "z": 16}},
    {"id": 5, "name": "XL-18", "weightMax": 70, "dimensions": {"x": 12, "y": 16, "z": 18}},
]

In [ ]:
# Sample order B: office / electronics (heavier, bulkier; likely 2+ boxes)
items = [
    {"id": 201, "name": "Monitor (boxed)", "color": "black", "weight": 14,
     "dimensions": {"x": 16, "y": 6, "z": 25}, "quantity": 1},
    {"id": 202, "name": "Keyboard (boxed)", "color": "slategray", "weight": 2.2,
     "dimensions": {"x": 2, "y": 6, "z": 18}, "quantity": 2},
    {"id": 203, "name": "Mouse (boxed)", "color": "silver", "weight": 0.4,
     "dimensions": {"x": 2.5, "y": 4, "z": 6}, "quantity": 3},
    {"id": 204, "name": "Paper Ream", "color": "white", "weight": 5,
     "dimensions": {"x": 2.25, "y": 8.5, "z": 11}, "quantity": 5},
    {"id": 205, "name": "Desk Lamp (boxed)", "color": "orange", "weight": 3.1,
     "dimensions": {"x": 5, "y": 6, "z": 15}, "quantity": 1},
    {"id": 206, "name": "Cable Pack", "color": "darkgray", "weight": 0.3,
     "dimensions": {"x": 1, "y": 3, "z": 5}, "quantity": 6},
]

boxes = [
    {"id": 1, "name": "M-12", "weightMax": 50, "dimensions": {"x": 8, "y": 12, "z": 12}},
    {"id": 2, "name": "Long-20", "weightMax": 50, "dimensions": {"x": 8, "y": 10, "z": 20}},
    {"id": 3, "name": "L-18", "weightMax": 60, "dimensions": {"x": 12, "y": 14, "z": 18}},
    {"id": 4, "name": "Monitor-28", "weightMax": 40, "dimensions": {"x": 18, "y": 8, "z": 28}},
    {"id": 5, "name": "XL-24", "weightMax": 70, "dimensions": {"x": 16, "y": 18, "z": 24}},
]

In [3]:
# Sample order C: nothing fits any box (expect 0 boxes, everything in leftovers)
# Rug, umbrella and headboard are too big in every rotation; the anvil fits
# by size but is heavier than any box's weight limit.
items = [
    {"id": 301, "name": "Rolled Rug", "color": "sienna", "weight": 12,
     "dimensions": {"x": 8, "y": 8, "z": 60}, "quantity": 1},
    {"id": 302, "name": "Patio Umbrella", "color": "darkgreen", "weight": 9,
     "dimensions": {"x": 6, "y": 6, "z": 52}, "quantity": 2},
    {"id": 303, "name": "Headboard", "color": "tan", "weight": 30,
     "dimensions": {"x": 4, "y": 40, "z": 58}, "quantity": 1},
    {"id": 304, "name": "Anvil", "color": "black", "weight": 120,
     "dimensions": {"x": 6, "y": 6, "z": 10}, "quantity": 1},
]

boxes = [
    {"id": 1, "name": "S-10", "weightMax": 40, "dimensions": {"x": 6, "y": 10, "z": 10}},
    {"id": 2, "name": "M-12", "weightMax": 50, "dimensions": {"x": 8, "y": 12, "z": 12}},
    {"id": 3, "name": "L-18", "weightMax": 60, "dimensions": {"x": 12, "y": 14, "z": 18}},
    {"id": 4, "name": "XL-24", "weightMax": 70, "dimensions": {"x": 16, "y": 18, "z": 24}},
]

In [4]:
from packer import pack
from planner import plan_with_validation
from prompt_builder import SYSTEM_PROMPT, build_user_prompt
from settings import (BlockRule, LoadLimit, PackingSettings, extract_settings,
                      final_plan, prepare_boxes, prepare_items, rules_text)

# Free-text instructions for this order. An LLM maps them onto
# PackingSettings (item, box and order rules, all applied and checked in code);
# everything else goes to the planner as soft instructions. Or set it directly.
INSTRUCTIONS = """Cut off should be divisible by 0.5, make sure to give instruction how much to cut it.
"""

settings = extract_settings(client, "gpt-5.4-nano", INSTRUCTIONS, items,
                            reasoning_effort="low")
# skip the LLM:
# settings = PackingSettings(padding=0.125, cut_step=0.5,
#                            blocks=[BlockRule(item_id=2, size=4)])  # stack by default;
#                            # arrangement="compact" or grid=[1, 2, 2] to change it
print(settings)

# item flags, padding, blocks, heavy-on-floor; boxes shrink by wall clearance
packed_items = prepare_items(items, boxes, settings)
packed_boxes = prepare_boxes(boxes, settings)
for item in packed_items:
    print(f"{item['id']}: {item['name']} ×{item['quantity']} "
          f"{item['dimensions']} {item['weight']:g} lb")

RULES = rules_text(packed_items, packed_boxes, settings)
USER_PROMPT = build_user_prompt(packed_items, packed_boxes,
                                "\n".join(settings.soft_instructions), RULES)
print(USER_PROMPT)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT}
]
print(messages)

padding=0.0 padding_by_item=[] blocks=[] upright=[] fixed_orientation=[] no_stack_on=[] max_load_on_top=[] floor_only_over_weight=None wall_clearance=0.0 cut_step=0.5 min_cut_height=None carrier_max_weight=None carrier_max_length=None carrier_max_girth=None min_fill=None max_boxes=None keep_together=[] keep_apart=[] balance_boxes=False soft_instructions=['make sure to give instruction how much to cut it.']
301: Rolled Rug ×1 {'x': 8.0, 'y': 8.0, 'z': 60.0} 12 lb
302: Patio Umbrella ×2 {'x': 6.0, 'y': 6.0, 'z': 52.0} 9 lb
303: Headboard ×1 {'x': 4.0, 'y': 40.0, 'z': 58.0} 30 lb
304: Anvil ×1 {'x': 6.0, 'y': 6.0, 'z': 10.0} 120 lb
Additional instructions:
make sure to give instruction how much to cut it.

items (4 rows)
id|name|color|weight|x|y|z|quantity
301|Rolled Rug|sienna|12|8|8|60|1
302|Patio Umbrella|darkgreen|9|6|6|52|2
303|Headboard|tan|30|4|40|58|1
304|Anvil|black|120|6|6|10|1

boxes (4 rows)
id|name|weightMax|x|y|z
1|S-10|40|6|10|10
2|M-12|50|8|12|12
3|L-18|60|12|14|18
4|XL-24

In [5]:
# Token usage and cost per run. Prices are USD per 1M tokens (input, output).
# Reasoning tokens are already inside the output count and billed as output.
PRICES = {
    "gpt-5.4-nano": (0.20, 1.25),
    "gpt-5.4-mini": (0.75, 4.50),
    "qwen/qwen3.8-27b:free": (0.0, 0.0),
    "llama3.2": (0.0, 0.0),  # local Ollama
}


def report(result, model):
    price_in, price_out = PRICES.get(model, (None, None))
    print(f"ok={result.ok} rounds={result.rounds} improved={result.improved} "
          f"plan from: {result.source}")
    print(f"tokens: input {result.prompt_tokens:,} | output {result.completion_tokens:,} "
          f"(reasoning {result.reasoning_tokens:,}) | total {result.total_tokens:,}")
    if price_in is None:
        print(f"cost: unknown, add {model!r} to PRICES")
    else:
        cost = (result.prompt_tokens * price_in
                + result.completion_tokens * price_out) / 1_000_000
        print(f"cost: ${cost:.4f}  (${price_in}/M in, ${price_out}/M out)")
    for note in (result.flat.notes if result.flat else []):
        print(f"note: {note}")

In [ ]:
# # OLLAMA (local)
# ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# result = plan_with_validation(ollama, "llama3.2", messages, items, boxes)
# report(result, "llama3.2")
# print("errors:", result.errors)
# print("hints:", result.hints)
# flat = result.flat
# plan = unflatten(flat, items, boxes)
# plan

# # Every plan is validated in code; errors go back until a plan passes
# result = plan_with_validation(
#     client, "gpt-5.4-mini", messages, items, boxes,
#     reasoning_effort="medium",
# )
# report(result, "gpt-5.4-mini")
# print("errors:", result.errors)
# print("hints:", result.hints)
# flat = result.flat
# plan = unflatten(flat, items, boxes)
# plan

In [6]:
# Code packer first (free, valid, low): target for the hints and fallback
baseline, baseline_errors = pack(packed_items, packed_boxes, settings)
print("code packer:", "valid" if not baseline_errors else baseline_errors[:2])

result = plan_with_validation(
    client, "gpt-5.4-nano", messages, packed_items, packed_boxes,
    settings=settings, baseline=None if baseline_errors else baseline,
    reasoning_effort="low",
)
report(result, "gpt-5.4-nano")
print("errors:", result.errors)
print("hints:", result.hints)
flat = result.flat
if not result.ok:
    # The validator rejected the last attempt: don't draw it as if it worked
    raise RuntimeError(f"No valid plan after {result.rounds} rounds: "
                       f"{len(result.errors)} errors (see above)")
plan = final_plan(flat, packed_items, packed_boxes, settings)
plan

code packer: valid
ok=True rounds=1 improved=False plan from: model
tokens: input 2,236 | output 232 (reasoning 20) | total 2,468
cost: $0.0007  ($0.2/M in, $1.25/M out)
errors: []
hints: []


Response(length_unit='in', weight_unit='lb', coordinate_system=CoordinateSystem(origin='inside_back_left_bottom', x_direction='bottom_to_top', y_direction='back_to_front', z_direction='left_to_right', position_reference='minimum_coordinate_corner'), boxes=[], leftover_items=[LeftoverItems(items=[Item(id=301, color='sienna', name='Rolled Rug', weight=12.0, dimensions=Dimensions(x=8.0, y=8.0, z=60.0), quantity=1)], reason='no allowed rotation of 8×8×60 fits any box (largest box side 24)'), LeftoverItems(items=[Item(id=302, color='darkgreen', name='Patio Umbrella', weight=9.0, dimensions=Dimensions(x=6.0, y=6.0, z=52.0), quantity=2)], reason='no allowed rotation of 6×6×52 fits any box (largest box side 24)'), LeftoverItems(items=[Item(id=303, color='tan', name='Headboard', weight=30.0, dimensions=Dimensions(x=4.0, y=40.0, z=58.0), quantity=1)], reason='no allowed rotation of 4×40×58 fits any box (largest box side 24)'), LeftoverItems(items=[Item(id=304, color='black', name='Anvil', weight

In [ ]:
flat = result.flat
plan = final_plan(flat, packed_items, packed_boxes, settings)
plan

In [ ]:
flat.model_dump()

In [7]:
from visualize import render_plan

plan = final_plan(flat, packed_items, packed_boxes, settings)
fig = render_plan(plan, path="plan.svg")

<Figure size 600x600 with 0 Axes>

In [ ]:
# Plotly: interactive (rotate/zoom/hover); WebGL depth buffer
from render_plotly import render_plotly

render_plotly(plan)

In [ ]:
# Isometric SVG: no extra dependencies, small file, real text labels
from IPython.display import SVG
from render_svg import render_svg

svg = render_svg(plan)
SVG(svg)

In [ ]:
# PyVista (VTK): lit off-screen render -> PIL image
from render_pyvista import render_pyvista

render_pyvista(plan)

In [ ]:
# vedo (VTK): same idea as PyVista with vedo's API -> PIL image
from render_vedo import render_vedo

render_vedo(plan)

In [ ]:
# three.js: self-contained HTML page with orbit controls and a step slider
from IPython.display import HTML
from render_threejs import render_threejs, threejs_iframe

page = render_threejs(plan)
HTML(threejs_iframe(page))

In [ ]:
# ## Sample cases (evaluation)
# One small order per rule. Each case: items, optional boxes, and either
# `settings` (set directly) or `text` (free text -> extract_settings).
# Run all with the cell below; costs tokens, one planner run per case.
from IPython.display import SVG
from render_svg import render_svg


def item(id, name, x, y, z, weight, quantity, color):
    return {"id": id, "name": name, "color": color, "weight": weight,
            "dimensions": {"x": x, "y": y, "z": z}, "quantity": quantity}


BOXES = [
    {"id": 1, "name": "Small", "weightMax": 65, "dimensions": {"x": 12, "y": 6, "z": 6.75}},
    {"id": 2, "name": "Medium", "weightMax": 80, "dimensions": {"x": 14.5, "y": 6.75, "z": 12.75}},
    {"id": 3, "name": "Large", "weightMax": 100, "dimensions": {"x": 16, "y": 12, "z": 12}},
]

CASES = {
    "basic": dict(items=items, boxes=boxes),
    "padding_cut": dict(items=[item(1, "Mug", 4, 3.5, 3.5, 0.8, 6, "teal")],
                        settings=PackingSettings(padding=0.125, cut_step=0.5)),
    "blocks": dict(items=[item(1, "Tape", 4.75, 4.75, 1, 0.3, 11, "blue")],
                   settings=PackingSettings(blocks=[BlockRule(item_id=1, size=4)])),
    "upright": dict(items=[item(1, "Bottle", 8, 3, 3, 1.2, 6, "orange")],
                    settings=PackingSettings(upright=[1])),
    "fixed": dict(items=[item(1, "Framed print", 10, 1, 8, 2, 2, "purple"),
                         item(2, "Tape", 4.75, 4.75, 1, 0.3, 4, "blue")],
                  settings=PackingSettings(fixed_orientation=[1])),
    "no_stack": dict(items=[item(1, "Glass vase", 6, 4, 4, 1.5, 1, "cyan"),
                            item(2, "Book", 1, 6, 9, 1, 4, "brown")],
                     settings=PackingSettings(no_stack_on=[1])),
    "max_load": dict(items=[item(1, "Carton", 4, 6, 10, 2, 1, "tan"),
                            item(2, "Can", 4, 3, 3, 1, 4, "gray")],
                     settings=PackingSettings(max_load_on_top=[LoadLimit(item_id=1, max_weight=2)])),
    "floor_only": dict(items=[item(1, "Dumbbell", 3, 4, 8, 25, 2, "black"),
                              item(2, "Pillow", 4, 6, 6, 0.5, 2, "pink")],
                       settings=PackingSettings(floor_only_over_weight=20)),
    "padding_by_item": dict(items=[item(1, "Monitor", 10, 3, 12, 8, 1, "black"),
                                   item(2, "Cable", 1, 3, 4, 0.2, 4, "gray")],
                            text="Pad the monitor with 0.5 in on every side, everything else 0.125 in."),
    "stock": dict(items=[item(1, "Tape", 4.75, 4.75, 1, 0.3, 20, "blue")],
                  boxes=[{**b, "stock": 0 if b["name"] == "Medium" else 1} for b in BOXES]),
    "wall_clearance": dict(items=[item(1, "Mug", 4, 3.5, 3.5, 0.8, 4, "teal")],
                           settings=PackingSettings(wall_clearance=1)),
    "carrier_limits": dict(items=[item(1, "Weight plate", 1, 9, 9, 10, 6, "black")],
                           settings=PackingSettings(carrier_max_weight=35, carrier_max_girth=60)),
    "min_cut": dict(items=[item(1, "Book", 1, 6, 9, 1, 2, "brown")],
                    settings=PackingSettings(cut_step=0.5, min_cut_height=4)),
    "min_fill": dict(items=[item(1, "Mug", 4, 3.5, 3.5, 0.8, 2, "teal")],
                     settings=PackingSettings(min_fill=0.6)),
    "max_boxes": dict(items=[item(1, "Pillow", 6, 10, 10, 0.5, 4, "pink")],
                      settings=PackingSettings(max_boxes=1)),
    "keep_together": dict(items=[item(1, "Kit base", 3, 5, 6, 2, 1, "green"),
                                 item(2, "Kit lid", 1, 5, 6, 0.5, 1, "olive"),
                                 item(3, "Tape", 4.75, 4.75, 1, 0.3, 8, "blue")],
                          settings=PackingSettings(keep_together=[[1, 2]])),
    "keep_apart": dict(items=[item(1, "Bleach", 9, 4, 4, 4, 1, "yellow"),
                              item(2, "Cereal", 10, 2.5, 7, 1, 1, "red")],
                       settings=PackingSettings(keep_apart=[[1, 2]])),
    "balance": dict(items=[item(1, "Weight plate", 1, 9, 9, 10, 4, "black")],
                    settings=PackingSettings(balance_boxes=True, carrier_max_weight=25)),
    "soft_instructions": dict(items=[item(1, "Dispenser", 9.5, 3.25, 4.5, 4, 1, "green"),
                                     item(2, "Tape", 4.75, 4.75, 1, 0.3, 6, "blue")],
                              text="Put the dispenser against the left wall."),
}

In [ ]:
def run_case(case, model="gpt-5.4-nano", **create_kwargs):
    """Whole pipeline for one case: settings -> prepare -> plan -> check."""
    boxes_in = case.get("boxes", BOXES)
    settings = case.get("settings") or PackingSettings()
    if case.get("text"):
        settings = extract_settings(client, model, case["text"], case["items"],
                                    reasoning_effort="low")
    items_p = prepare_items(case["items"], boxes_in, settings)
    boxes_p = prepare_boxes(boxes_in, settings)
    prompt = build_user_prompt(items_p, boxes_p,
                               "\n".join(settings.soft_instructions),
                               rules_text(items_p, boxes_p, settings))
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}]
    baseline, baseline_errors = pack(items_p, boxes_p, settings)
    result = plan_with_validation(client, model, msgs, items_p, boxes_p,
                                  settings=settings,
                                  baseline=None if baseline_errors else baseline,
                                  **create_kwargs)
    plan = (final_plan(result.flat, items_p, boxes_p, settings)
            if result.ok else None)
    return result, plan, settings


def summary_row(name, result, model):
    price_in, price_out = PRICES.get(model, (0, 0))
    cost = (result.prompt_tokens * price_in
            + result.completion_tokens * price_out) / 1_000_000
    flat = result.flat
    return {
        "case": name, "ok": result.ok, "from": result.source,
        "rounds": result.rounds,
        "boxes": len(flat.boxes) if flat else "-",
        "leftover": sum(r.quantity for r in flat.leftovers) if flat else "-",
        "tokens": result.total_tokens, "cost": f"${cost:.4f}",
        "first error": result.errors[0][:60] if result.errors else "",
    }

In [ ]:
# Run every case (or a subset: for name in ["upright", "no_stack"]: ...)
MODEL = "gpt-5.4-nano"
RESULTS, PLANS = {}, {}
for name, case in CASES.items():
    result, plan, used = run_case(case, MODEL, reasoning_effort="low")
    RESULTS[name], PLANS[name] = result, plan
    print(f"{name}: ok={result.ok} rounds={result.rounds}")

rows = [summary_row(n, r, MODEL) for n, r in RESULTS.items()]
header = list(rows[0])
print("| " + " | ".join(header) + " |")
print("|" + "---|" * len(header))
for row in rows:
    print("| " + " | ".join(str(row[h]) for h in header) + " |")

In [ ]:
# Look at one case
name = "upright"
print(PLANS[name].notes if PLANS[name] else RESULTS[name].errors)
SVG(render_svg(PLANS[name])) if PLANS[name] else None